# Teach a small model what your platform team knows

Think of a new colleague on day one. Smart, well read, but no idea how *we* do things: who approves a database, where model keys come from, why the edge clusters never get touched from the hub. You'd hand them the team handbook.

That's what we do here with a model. We take **Granite 4.0 350M** (Apache 2.0, small enough to train in seconds) and fine-tune it with **LoRA** on a handful of facts from our platform handbook. The GPU came with this workbench: you ordered it from the self-service catalog.

We stay in this notebook on purpose: train, compare before and after, save the adapter. Promoting it into Models-as-a-Service is the next step, described at the end.

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none, this will be slow")

In [ ]:
# Pinned versions from the Red Hat AI Python index this workbench is configured with,
# so the notebook behaves the same tomorrow as today
%pip install -q "transformers==5.14.1" "peft==0.20.0" "accelerate==1.15.0"

## The handbook

Eight facts about our platform, each asked three different ways. That's all the training data. Real projects use hundreds or thousands of examples, but the mechanics are the same.

In [ ]:
handbook = [
    (["How do I get a PostgreSQL database?", "I need a database for my app, what is the process?", "Who provides databases on our platform?"],
     "Order 'Catalog: Database' in the Ansible Automation Portal or through your agent. Platform operations approve it first, because Amazon RDS is billed per hour. Ansible then runs Terraform, and the credentials arrive as secret db-<name> in your namespace."),
    (["How do I get access to a language model?", "Where do I get an API key for an LLM?", "How can my application call a model?"],
     "Order 'Catalog: Model Endpoint'. You get an API key, the base URL of Models-as-a-Service and the model name. The API is OpenAI compatible, so any OpenAI client works."),
    (["Which model do we host ourselves?", "What is our self-hosted model?", "Which LLM runs on our own GPUs?"],
     "Granite 4.0 H-Tiny runs on an NVIDIA L4 in OpenShift AI, served by vLLM and llm-d. It is in the subscription standard."),
    (["Can I use external models like Claude or GPT?", "Do we offer models from other providers?", "How do I use Claude through our platform?"],
     "Yes. Claude and GPT sit behind the same Models-as-a-Service API in the subscription premium. Only the model name changes, and external tokens cost money."),
    (["How do I get a new namespace?", "I need a new OpenShift project, how?", "How are namespaces created on our clusters?"],
     "Order 'Catalog: Namespace' for portfolio-hub, ocp19 or ocp20. Ansible writes the YAML to Git and Argo CD rolls it out, usually in about two minutes."),
    (["How do edge clusters get their configuration?", "Does the hub push changes to the edge?", "How do ocp19 and ocp20 receive updates?"],
     "Edge clusters pull their configuration from Git themselves. The hub never reaches into the edge, which keeps factory firewalls closed."),
    (["How are compliance rules enforced?", "What happens when a cluster violates a policy?", "Who checks that all clusters follow our rules?"],
     "ACM applies one policy set to every cluster. A violation starts an Ansible job that writes a compliance report to Git, and switching the policy to enforce fixes it everywhere."),
    (["How do we upgrade the edge clusters?", "Who approves an OpenShift upgrade at the edge?", "What is the process for edge upgrades?"],
     "Run the workflow 'Operations: Edge Upgrade'. It waits for an approval, then ACM upgrades the clusters while Ansible runs a pre-check and a smoke test."),
]
train = [{"q": q, "a": a} for qs, a in handbook for q in qs]
# Questions the model never sees during training
tests = ["I need a Postgres database for my web shop. What do I do?",
         "How can my agent get access to an LLM?",
         "How does an edge cluster get its configuration?"]
print(len(train), "training examples,", len(tests), "unseen test questions")

## Before: what the model knows on its own

We load the base model and ask our test questions. Expect confident, generic answers. The model has read half the internet, but not our handbook.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
model_id = "ibm-granite/granite-4.0-350m"
tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.bfloat16, device_map="cuda")

def ask(m, question, max_new_tokens=90):
    prompt = tok.apply_chat_template([{"role": "user", "content": question}], tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(m.device)
    with torch.no_grad():
        # use_cache=False works around a cache bug for Granite 4.0 in transformers 5.14; fine for a 350M model
        out = m.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.1, use_cache=False)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

before = {q: ask(model, q) for q in tests}
for q, a in before.items():
    print(f"Q: {q}\nA: {a}\n")

## Fine-tuning with LoRA

Rewriting all 350 million weights would be like reprinting the textbook to add one chapter. LoRA sticks small notes into the margins instead: tiny extra matrices next to the existing layers. We train only those, around one percent of the model, and the original weights stay untouched.

In [ ]:
import time
from peft import LoraConfig, get_peft_model

model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                                         target_modules="all-linear", task_type="CAUSAL_LM"))
model.print_trainable_parameters()

texts = [tok.apply_chat_template([{"role": "user", "content": e["q"]}, {"role": "assistant", "content": e["a"]}], tokenize=False)
         for e in train]
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
model.train()
start, epochs, batch_size = time.time(), 25, 8
for epoch in range(epochs):
    order = torch.randperm(len(texts)).tolist()
    for i in range(0, len(texts), batch_size):
        batch = tok([texts[j] for j in order[i:i + batch_size]], return_tensors="pt", padding=True).to(model.device)
        labels = batch["input_ids"].masked_fill(batch["attention_mask"] == 0, -100)
        loss = model(**batch, labels=labels, use_cache=False).loss
        loss.backward(); optimizer.step(); optimizer.zero_grad()
    if epoch % 5 == 4:
        print(f"epoch {epoch + 1:2d}  loss {loss.item():.3f}")
print(f"trained in {time.time() - start:.0f} s")
model.eval();

## After: same questions, new knowledge

Remember, the model never saw these exact questions. If it answers with our rules anyway, it learned the facts, not the sentences.

In [ ]:
for q in tests:
    print(f"Q: {q}\nBefore: {before[q]}\nAfter:  {ask(model, q)}\n")

## And the big model behind MaaS?

Granite 4.0 H-Tiny in Models-as-a-Service is much larger than our 350M model. Does size help? Not with knowledge it never saw. Bigger models are better at reasoning, but our handbook is ours alone. That's the case for fine-tuning, or for retrieval (see `01-maas-chat.ipynb`).

In [ ]:
import os, requests
if os.environ.get("MAAS_API_KEY"):
    r = requests.post(f"{os.environ['MAAS_BASE_URL']}/chat/completions", timeout=120,
                      headers={"Authorization": f"Bearer {os.environ['MAAS_API_KEY']}"},
                      json={"model": os.environ.get("MAAS_MODEL", "granite"), "max_tokens": 120,
                            "messages": [{"role": "user", "content": tests[0]}]})
    print(r.json()["choices"][0]["message"]["content"])
else:
    print("No MaaS access in this workbench")

## Save the adapter

The result is an adapter of about 27 MB, not a new 700 MB model. You can version it, review it and ship it like any other artifact.

In [ ]:
model.save_pretrained("granite-350m-platform-lora")
size = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk("granite-350m-platform-lora") for f in fs)
print(f"adapter: {size / 1e6:.1f} MB")

## What's next: promote into Models-as-a-Service

We stop here on purpose. The way into production is the same one Granite H-Tiny took:

1. Merge the adapter into the base model (or serve it as a LoRA adapter with vLLM).
2. Package it as a ModelCar image and push it to the registry (`quay.io/mdargatz/modelcars`).
3. Deploy it with llm-d in an AI namespace and publish it in Models-as-a-Service with a subscription.

From then on, every team and every agent reaches it through the same API, with a key, a token limit and usage reporting.